In [2]:
import pandas as pd
import joblib

In [7]:
# Load data
df = pd.read_csv("../data/processed/modeling_dataset.csv")

# Load models
churn_model = joblib.load("../models/churn_model.joblib")
purchase_model = joblib.load("../models/purchase_model.joblib")

# Load scalers
churn_scaler = joblib.load("../models/churn_scaler.joblib")
purchase_scaler = joblib.load("../models/purchase_scaler.joblib")

# Columns dropped during training (important!)
purchase_drop_cols = joblib.load("../models/purchase_drop_cols.joblib")

In [8]:
df.head()

,frequency,monetary,avg_order_value,total_interactions,interaction_types_count,engagement_segment,engaged_not_purchased,purchase_frequency_rate,interaction_purchase_ratio,customer_segment,location,acquisition_channel,age,gender,is_churned,will_purchase
0,31,482863.71,15576.248710,335,4,High Engagement,0,0,10,medium,Bhubaneswar,ads,25,male,0,1
1,45,820236.81,18227.484667,389,4,High Engagement,0,0,8,high,Hyderabad,referral,47,female,0,1
2,0,0.00,0.000000,8,4,Medium Engagement,1,0,0,low,Ahmedabad,organic,46,female,0,0
3,31,537397.24,17335.394839,319,4,High Engagement,0,0,10,medium,Bangalore,organic,38,male,0,1
4,95,1556310.10,16382.211579,752,4,High Engagement,0,0,7,high,Hyderabad,referral,37,female,0,1


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   frequency                   8000 non-null   int64  
 1   monetary                    8000 non-null   float64
 2   avg_order_value             8000 non-null   float64
 3   total_interactions          8000 non-null   int64  
 4   interaction_types_count     8000 non-null   int64  
 5   engagement_segment          8000 non-null   object 
 6   engaged_not_purchased       8000 non-null   int64  
 7   purchase_frequency_rate     8000 non-null   int64  
 8   interaction_purchase_ratio  8000 non-null   int64  
 9   customer_segment            8000 non-null   object 
 10  location                    8000 non-null   object 
 11  acquisition_channel         8000 non-null   object 
 12  age                         8000 non-null   int64  
 13  gender                      8000 

In [ ]:
# ---------------------------
# Step 1: Prepare Features
# ---------------------------

# Drop target columns
X = df.drop(columns=["is_churned", "will_purchase"])

# Separate categorical columns
categorical_cols = X.select_dtypes(include=["object"]).columns

# One-hot encode
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)


# --------------------------------
# Step 2: Align with CHURN MODEL
# --------------------------------

churn_features = churn_scaler.feature_names_in_

# Add missing columns
for col in churn_features:
    if col not in X_encoded.columns:
        X_encoded[col] = 0

# Ensure same order
X_churn = X_encoded[churn_features]

# Scale + predict
X_churn_scaled = churn_scaler.transform(X_churn)
df["churn_prob"] = churn_model.predict_proba(X_churn_scaled)[:, 1]


# -----------------------------------
# Step 3: Align with PURCHASE MODEL
# -----------------------------------

purchase_features = purchase_scaler.feature_names_in_

# Add missing columns
for col in purchase_features:
    if col not in X_encoded.columns:
        X_encoded[col] = 0

# Align
X_purchase = X_encoded[purchase_features]

# Scale + predict
X_purchase_scaled = purchase_scaler.transform(X_purchase)
df["purchase_prob"] = purchase_model.predict_proba(X_purchase_scaled)[:, 1]


# -------
# Check
# -------
df[["churn_prob", "purchase_prob"]].head()

,churn_prob,purchase_prob
0,1.534636e-06,0.999965
1,9.047102e-09,0.999988
2,6.218583e-01,0.374260
3,1.283296e-06,0.999967
4,1.029391e-16,1.000000


In [17]:
# Expected Revenue
df["expected_revenue"] = df["purchase_prob"] * df["avg_order_value"]

# Adjust for Churn Risk
df["adjusted_revenue"] = df["expected_revenue"] * (1 - df["churn_prob"])

# Final Score
df["final_score"] = df["adjusted_revenue"]

# Quick check
df[["purchase_prob", "churn_prob", "avg_order_value", "final_score"]].head()

,purchase_prob,churn_prob,avg_order_value,final_score
0,0.999965,1.534636e-06,15576.248710,15575.673854
1,0.999988,9.047102e-09,18227.484667,18227.261961
2,0.374260,6.218583e-01,0.000000,0.000000
3,0.999967,1.283296e-06,17335.394839,17334.804631
4,1.000000,1.029391e-16,16382.211579,16382.211560


In [18]:
# Budget Setup
cost_per_customer = 50
budget = 50000

max_customers = int(budget / cost_per_customer)

# Rank Customers
df = df.sort_values(by="final_score", ascending=False).reset_index(drop=True)

# Select Top Customers
df["selected"] = 0
df.loc[:max_customers - 1, "selected"] = 1

# Quick check
df[["final_score", "selected"]].head(10)

,final_score,selected
0,91419.989549,1
1,58380.466439,1
2,50682.062998,1
3,47741.537011,1
4,47150.757301,1
5,44356.373084,1
6,43190.271706,1
7,43148.543165,1
8,42178.094838,1
9,37941.558978,1


In [ ]:
target_df = df[df["selected"] == 1]

uplift_factor = 0.1

incremental_revenue = (
    target_df["purchase_prob"] *
    target_df["avg_order_value"] *
    uplift_factor
)

total_expected_revenue = incremental_revenue.sum()
total_cost = target_df.shape[0] * cost_per_customer
net_profit = total_expected_revenue - total_cost


print("Total Target Customers:", target_df.shape[0])
print("Incremental Revenue: ₹", round(total_expected_revenue, 2))
print("Total Campaign Cost: ₹", total_cost)
print("Net Profit: ₹", round(net_profit, 2))

Total Target Customers: 1000
Incremental Revenue: ₹ 1666141.71
Total Campaign Cost: ₹ 50000
Net Profit: ₹ 1616141.71


## Optimization Summary

### Objective

To identify the most valuable customers to target under a fixed marketing budget, using model predictions and business-driven scoring.

### Approach

* Used trained models to predict:
  * Customer churn probability
  * Purchase probability
* Computed **expected customer value** using:
  * Purchase likelihood
  * Average order value
  * Churn risk adjustment
* Introduced a **conservative uplift factor (10%)** to estimate **incremental revenue**, ensuring realistic business assumptions.

### Targeting Strategy

* Ranked customers based on final value score
* Applied budget constraint:
  * Cost per customer: ₹50
  * Total budget: ₹50,000
  * Maximum customers targeted: 1,000
* Selected top 1,000 high-value customers

### Business Impact

* Estimated Incremental Revenue: ₹16.66 Lakhs
* Campaign Cost: ₹50,000
* Net Expected Profit: ₹16.16 Lakhs

### Key Insight

A small, targeted subset of high-value customers can generate significant returns when guided by predictive modeling and value-based prioritization.

### Conclusion

This approach demonstrates how machine learning predictions can be translated into actionable business decisions, enabling efficient resource allocation and maximizing marketing ROI.